In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
np.random.seed(42)
tf.random.set_seed(42)

In [4]:
df = pd.read_csv("../data/pontianak_weather_daily_2021_2024.csv")
df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y")

In [5]:
df = df.drop_duplicates(subset="date").set_index("date").sort_index()
df = df.asfreq("D") 

In [6]:
df[["TAVG", "RH_AVG"]] = df[["TAVG", "RH_AVG"]].interpolate(limit_direction="both")

data = df[["TAVG", "RH_AVG"]].values.astype("float32")
dates = df.index

In [7]:
train_end = np.where(dates == pd.Timestamp("2024-01-01"))[0][0]
train_raw = data[:train_end]
test_raw = data[train_end:]

In [8]:
mean, std = train_raw.mean(axis=0), train_raw.std(axis=0)
train_s = (train_raw - mean) / std
test_s = (test_raw - mean) / std

In [10]:
WINDOW = 14

def make_windows(arr, window):
    X, y = [], []
    for i in range(len(arr) - window):
        X.append(arr[i : i + window])        
        y.append(arr[i + window, 0])         
    return np.array(X), np.array(y)

In [11]:
X_train, y_train = make_windows(train_s, WINDOW)
X_test, y_test = make_windows(np.vstack([train_s[-WINDOW:], test_s]), WINDOW)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: (1081, 14, 2) X_test: (366, 14, 2)


In [12]:
model = keras.Sequential([
    keras.Input(shape=(WINDOW, 2)),
    keras.layers.SimpleRNN(32),
    keras.layers.Dense(1),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)               │ (None, 32)                  │           1,120 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,153 (4.50 KB)

 Trainable params: 1,153 (4.50 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
early = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)
history = model.fit(
    X_train, y_train,
    validation_split=0.15,   
    epochs=100, batch_size=32, shuffle=False,  
    callbacks=[early], verbose=0,
)
print("Eğitilen epoch sayısı:", len(history.history["loss"]))

Eğitilen epoch sayısı: 30


In [14]:
def inv_temp(z):
    return z * std[0] + mean[0]

pred = inv_temp(model.predict(X_test, verbose=0).ravel())
true = inv_temp(y_test)

persist = inv_temp(X_test[:, -1, 0])
window_mean = inv_temp(X_test[:, :, 0].mean(axis=1))

In [15]:
def report(name, y_true, y_hat):
    mae = mean_absolute_error(y_true, y_hat)
    rmse = np.sqrt(mean_squared_error(y_true, y_hat))
    print(f"{name:<22} MAE={mae:.3f} °C   RMSE={rmse:.3f} °C")

print("\n--- 2024 test sonuçları ---")
report("Persistence (dünkü)", true, persist)
report("14 gün ortalaması", true, window_mean)
report("Simple RNN", true, pred)


--- 2024 test sonuçları ---
Persistence (dünkü)    MAE=0.866 °C   RMSE=1.101 °C
14 gün ortalaması      MAE=0.812 °C   RMSE=0.999 °C
Simple RNN             MAE=0.787 °C   RMSE=0.969 °C


In [16]:
model.save("../model/weather_rnn.keras")